# Chapter 6 -- Loop Engineering (Solved)

Work through this notebook **after reading** `notes/ch06-loop-engineering.md`. This chapter wraps Chapter 5's `Harness` in a **verified loop**: a real verifier (actually running a test suite, not a scripted judge), oscillation detection, a budget guard, and a printed loop ledger -- against a genuinely real task, a small buggy module that needs to be fixed until its test suite passes.

Two exercises below have a stub to fill in: **no-progress detection** and **best-of-3 with verifier-based selection**. Everything is verified offline -- no API key needed for either exercise.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline -- this cell")
        print("only matters for the optional real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## The Shared Task: a Real, Genuinely Broken Module

`V1_BUGGY` below has three functions, two of which have a real bug. `TESTS` checks all three. `run_tests` is a **real** verifier -- it actually `exec()`s the code and runs the test functions against it (notes Section 2's ladder: this implements the "exit codes" rung via the `exec()` try/except, and the "tests" rung via the assertions). Nothing about verification in this notebook is scripted or faked -- only the *model's proposed fixes*, later, are scripted for determinism.

In [ ]:
TESTS = [
    ("test_add", lambda ns: ns["add"](2, 3) == 5),
    ("test_is_even", lambda ns: ns["is_even"](4) is True and ns["is_even"](3) is False),
    ("test_clamp", lambda ns: ns["clamp"](15, 0, 10) == 10 and ns["clamp"](-5, 0, 10) == 0),
]

V1_BUGGY = """
def add(a, b):
    return a - b

def is_even(n):
    return n % 2 == 1

def clamp(x, lo, hi):
    return max(lo, min(x, hi))
"""


def run_tests(code, tests):
    """
    A REAL verifier -- Section 2's ladder, two rungs deep:
    rung 1 (exit codes): does `code` even exec() without raising?
    rung 2 (tests): run each test function against the resulting namespace.

    Returns: (syntax_ok, passed_names, failed_names, error_or_None)
    """
    namespace = {}
    try:
        exec(code, namespace)
    except Exception as exc:
        return False, [], [name for name, _ in tests], f"{type(exc).__name__}: {exc}"

    passed, failed = [], []
    for name, test_fn in tests:
        try:
            ok = test_fn(namespace)
        except Exception:
            ok = False
        (passed if ok else failed).append(name)
    return True, passed, failed, None


ok, passed, failed, err = run_tests(V1_BUGGY, TESTS)
print(f"V1_BUGGY: syntax_ok={ok}  passed={passed}  failed={failed}")
print(f"{len(failed)} of {len(TESTS)} tests currently failing -- this is the task the loop below has to fix.")


## The `submit_patch` Tool and a Scripted Fake Client

The "model" proposes a fix by calling `submit_patch(code=...)` with a full replacement for the module. Same offline-testing approach as every prior chapter: `FakeClient` mimics `.messages.create()`'s exact response shape, so the loop's real logic (verification, oscillation detection, the ledger) is exercised deterministically with no API key.

In [ ]:
from types import SimpleNamespace

SUBMIT_PATCH_SCHEMA = {
    "name": "submit_patch",
    "description": "Submit a full replacement for the module's source code. It will be run against the test suite immediately.",
    "input_schema": {
        "type": "object",
        "properties": {"code": {"type": "string", "description": "The complete, updated module source code"}},
        "required": ["code"],
    },
}


def _text(text):
    return SimpleNamespace(type="text", text=text)


def _tool_use(tool_id, name, tool_input):
    return SimpleNamespace(type="tool_use", id=tool_id, name=name, input=tool_input)


def _response(stop_reason, content):
    return SimpleNamespace(stop_reason=stop_reason, content=content, usage=SimpleNamespace(input_tokens=100, output_tokens=20))


class FakeClient:
    """Plays back a scripted list of responses, ignoring the actual request -- same pattern as every prior chapter."""

    def __init__(self, script):
        self.script = list(script)
        self.messages = SimpleNamespace(create=self._create)

    def _create(self, **kwargs):
        if not self.script:
            raise RuntimeError("FakeClient script exhausted -- the loop asked for more steps than scripted")
        return self.script.pop(0)


print("submit_patch tool and FakeClient defined.")


## The Verified Loop (Given)

`run_verified_loop` is this chapter's core deliverable: notes Section 1's `trigger -> act -> verify -> stop` cycle, with three specific things baked in that a naive loop skips:

1. **It never trusts `end_turn` alone** (notes Section 3) -- even if the model declares itself done, the loop re-checks the real gates (`failed` from the real verifier) before agreeing. A premature declaration is rejected and the loop continues.
2. **Oscillation detection** (notes Section 5) -- if the exact same set of failing tests repeats for `oscillation_window` consecutive iterations, the loop stops rather than retrying a strategy that provably isn't working.
3. **A printed ledger**, one line per iteration, in the same shape notes' own roadmap specified: `iter N | verify=... | progress=... | <action>`.

In [ ]:
def run_verified_loop(client, model, initial_code, tests, max_iters=10, oscillation_window=3, no_progress_window=3):
    """
    trigger -> act -> verify -> stop, with real verification and real
    oscillation detection. Returns (success, iterations_taken, final_code).
    """
    current_code = initial_code
    messages = [{"role": "user", "content": "Fix the module so all tests pass."}]
    ok, passed, failed, err = run_tests(current_code, tests)
    progress_history = [len(passed)]
    fingerprint_history = []

    if not failed:
        print("iter  0 | verify=PASS         | progress=n/a | SUCCESS -- all gates passed (nothing to fix)")
        return True, 0, current_code

    for i in range(1, max_iters + 1):
        response = client.messages.create(model=model, max_tokens=1024, tools=[SUBMIT_PATCH_SCHEMA], messages=messages)

        if response.stop_reason == "end_turn":
            # Victory-declaration check (notes Section 3): never trust end_turn alone.
            if failed:
                print(f"iter {i:2d} | model declared done, but {len(failed)} test(s) still fail: {failed} -- REJECTED, continuing")
                messages.append({"role": "assistant", "content": response.content})
                messages.append({"role": "user", "content": f"Not done -- {len(failed)} test(s) still fail: {failed}"})
                continue
            print(f"iter {i:2d} | verify=PASS         | progress=n/a | SUCCESS -- all gates passed")
            return True, i, current_code

        tool_block = next(b for b in response.content if b.type == "tool_use")
        current_code = tool_block.input["code"]
        ok, passed, failed, err = run_tests(current_code, tests)
        progress = len(passed) - progress_history[-1]
        progress_history.append(len(passed))
        fingerprint_history.append(frozenset(failed))

        verify_str = "PASS" if not failed else f"FAIL({len(failed)} tests)"

        if not failed:
            print(f"iter {i:2d} | verify={verify_str:12s} | progress={progress:+d} | SUCCESS -- all gates passed")
            return True, i, current_code

        oscillating = (len(fingerprint_history) >= oscillation_window
                       and len(set(fingerprint_history[-oscillation_window:])) == 1)
        no_progress = (len(progress_history) > no_progress_window
                       and progress_history[-1] <= progress_history[-1 - no_progress_window])

        action = "retry with error ctx"
        if oscillating:
            action = "STOP -- oscillation detected"
        elif no_progress:
            action = "STOP -- no progress"

        print(f"iter {i:2d} | verify={verify_str:12s} | progress={progress:+d} | {action}")

        if oscillating or no_progress:
            return False, i, current_code

        messages.append({"role": "assistant", "content": response.content})
        messages.append({
            "role": "user",
            "content": [{"type": "tool_result", "tool_use_id": tool_block.id,
                         "content": f"{len(failed)} test(s) failing: {failed}"}],
        })

    print(f"iter {max_iters:2d} | STOP -- budget (max_iters) exceeded")
    return False, max_iters, current_code


print("run_verified_loop defined.")


## Three Scripted Trajectories

Same underlying task, three different scripted "models," each exercising a different piece of the loop for real:

- **A) A model that actually fixes it** -- one bug per patch, 2 real iterations, clean success.
- **B) A model stuck in a loop** -- resubmits the exact same broken code repeatedly. Watch the oscillation detector stop it well before `max_iters`.
- **C) A model that declares victory early** -- calls `end_turn` while tests still fail. Watch the loop reject it and keep going, then genuinely succeed two steps later.

In [ ]:
V2_FIX_ADD = """
def add(a, b):
    return a + b

def is_even(n):
    return n % 2 == 1

def clamp(x, lo, hi):
    return max(lo, min(x, hi))
"""

V3_FIX_BOTH = """
def add(a, b):
    return a + b

def is_even(n):
    return n % 2 == 0

def clamp(x, lo, hi):
    return max(lo, min(x, hi))
"""

SUCCESS_SCRIPT = [
    _response("tool_use", [_tool_use("c1", "submit_patch", {"code": V2_FIX_ADD})]),
    _response("tool_use", [_tool_use("c2", "submit_patch", {"code": V3_FIX_BOTH})]),
    _response("end_turn", [_text("All tests pass now.")]),
]

print("-" * 60)
print("SCENARIO A: a model that actually fixes the bugs")
print("-" * 60)
success, iters, final_code = run_verified_loop(FakeClient(SUCCESS_SCRIPT), "fake-model", V1_BUGGY, TESTS)
assert success is True, "Scenario A should succeed"
assert iters == 2, f"expected exactly 2 real iterations, got {iters}"
_, final_passed, final_failed, _ = run_tests(final_code, TESTS)
assert not final_failed, "the final code should pass every test"
print(f"\nConfirmed: succeeded in {iters} iterations, all {len(final_passed)} tests passing.")


In [ ]:
OSCILLATE_SCRIPT = [
    _response("tool_use", [_tool_use("c1", "submit_patch", {"code": V1_BUGGY})]),
    _response("tool_use", [_tool_use("c2", "submit_patch", {"code": V1_BUGGY})]),
    _response("tool_use", [_tool_use("c3", "submit_patch", {"code": V1_BUGGY})]),
    _response("tool_use", [_tool_use("c4", "submit_patch", {"code": V1_BUGGY})]),
]

print("-" * 60)
print("SCENARIO B: a model stuck resubmitting the same broken code")
print("-" * 60)
success, iters, final_code = run_verified_loop(FakeClient(OSCILLATE_SCRIPT), "fake-model", V1_BUGGY, TESTS, max_iters=10)
assert success is False, "Scenario B should NOT succeed"
assert iters == 3, f"expected the oscillation detector to stop this at iteration 3, got {iters}"
print(f"\nConfirmed: oscillation detector stopped the loop after {iters} iterations,")
print(f"well short of the max_iters=10 budget -- exactly what Section 5's mechanism is for.")


In [ ]:
PREMATURE_SCRIPT = [
    _response("end_turn", [_text("I believe this is done.")]),
    _response("tool_use", [_tool_use("c1", "submit_patch", {"code": V2_FIX_ADD})]),
    _response("tool_use", [_tool_use("c2", "submit_patch", {"code": V3_FIX_BOTH})]),
    _response("end_turn", [_text("Now all tests pass.")]),
]

print("-" * 60)
print("SCENARIO C: a model that declares victory before the gates pass")
print("-" * 60)
success, iters, final_code = run_verified_loop(FakeClient(PREMATURE_SCRIPT), "fake-model", V1_BUGGY, TESTS)
assert success is True, "Scenario C should eventually succeed"
assert iters == 3, f"expected 3 iterations (1 rejected + 2 real fixes), got {iters}"
print(f"\nConfirmed: the premature end_turn was rejected, not trusted -- the loop only")
print(f"declared success once the real verifier actually reported zero failures.")


## Exercise 1 -- No-Progress Detection

Implement `detect_no_progress(progress_history, window=3)`: notes Section 6's concrete signal, independent of oscillation. Return `True` if the passing-test count has **not improved** over the last `window` iterations (i.e. the count `window` steps ago is greater than or equal to the current count), `False` otherwise. `progress_history` is a list of passing-test counts, one entry per iteration, oldest first.

In [ ]:
def detect_no_progress(progress_history, window=3):
    """
    Return True if progress_history[-1] has not improved over
    progress_history[-1 - window] (i.e. stuck for `window` iterations).
    Return False if there isn't yet enough history to judge (len < window + 1).
    """
    if len(progress_history) <= window:
        return False
    return progress_history[-1] <= progress_history[-1 - window]


In [ ]:
assert detect_no_progress([1, 1, 1, 1], window=3) is True, "stuck at 1 for 4 iterations should be flagged"
assert detect_no_progress([1, 2, 3], window=3) is False, "steadily improving should not be flagged"
assert detect_no_progress([0, 1, 1, 1, 1], window=3) is True, "stuck for the last 3 -- the earlier 0->1 jump has aged out of the window"
assert detect_no_progress([1, 1], window=3) is False, "not enough history yet -- must not false-positive"
assert detect_no_progress([1, 1, 2], window=2) is False, "improved within the window -- should not be flagged"

print("Exercise 1 PASSED -- detect_no_progress correctly flags a stalled")
print("passing-test count and does not false-positive on real improvement")
print("or on insufficient history.")


## Exercise 2 -- Best-of-3 With Verifier-Based Selection

Implement `best_of_n(candidates, tests)`: notes Section 11's best-of-N, using the **real** `run_tests` verifier (not a vibe, not a coin flip) to pick a winner. Given a list of candidate code strings, run the verifier against each and return `(best_index, passed_names, failed_names)` for whichever candidate has the most passing tests. Break ties by earliest index.

In [ ]:
def best_of_n(candidates, tests):
    """
    Run run_tests against every candidate; return (best_index, passed, failed)
    for the candidate with the most passing tests. Ties go to the earliest index.
    """
    results = []
    for i, candidate_code in enumerate(candidates):
        ok, passed, failed, err = run_tests(candidate_code, tests)
        results.append((i, len(passed), passed, failed))
    best = max(results, key=lambda r: r[1])
    return best[0], best[2], best[3]


In [ ]:
V_FIX_ONLY_IS_EVEN = """
def add(a, b):
    return a - b

def is_even(n):
    return n % 2 == 0

def clamp(x, lo, hi):
    return max(lo, min(x, hi))
"""

candidates = [V2_FIX_ADD, V_FIX_ONLY_IS_EVEN, V3_FIX_BOTH]
best_idx, passed, failed = best_of_n(candidates, TESTS)

assert best_idx == 2, f"expected the fully-correct candidate (index 2) to win, got index {best_idx}"
assert failed == [], f"the winning candidate should have zero failing tests, got {failed}"
assert len(passed) == 3

# a case with no clear winner among two equally-good candidates -- earliest index wins
tie_candidates = [V2_FIX_ADD, V_FIX_ONLY_IS_EVEN]
tie_idx, tie_passed, tie_failed = best_of_n(tie_candidates, TESTS)
assert tie_idx == 0, "a tie should resolve to the earliest index"

print("Exercise 2 PASSED -- best_of_n correctly picks the candidate with the most")
print("passing tests (using the real verifier, not a guess), and ties resolve")
print("to the earliest candidate.")


## Given: Best-of-3 in Action

Three independently-proposed fixes for the same remaining bug -- only one of them is actually fully correct. `best_of_n` (your implementation) picks it using the real verifier, not by asking a judge to guess.

In [ ]:
print("-" * 60)
print("BEST-OF-3: three candidate patches, one real verifier")
print("-" * 60)
for i, cand in enumerate(candidates):
    _, cand_passed, cand_failed, _ = run_tests(cand, TESTS)
    print(f"  candidate {i}: passed={cand_passed}  failed={cand_failed}")

winner_idx, winner_passed, winner_failed = best_of_n(candidates, TESTS)
print(f"\nWinner: candidate {winner_idx} (passed={len(winner_passed)}/{len(TESTS)} tests)")


## Notes Section 12, Made Concrete: Verification Beats Better Prompting

The reliability arithmetic from the notes, reproduced as code: a 20-step chain at $p=0.95$ per step, unverified, versus the same chain with a verifier catching a fraction $c$ of failures and allowing one retry.

In [ ]:
p = 0.95
q = 1 - p
n_steps = 20
baseline = p ** n_steps

print("-" * 60)
print("RELIABILITY ARITHMETIC: verification beats better prompting")
print("-" * 60)
print(f"Unverified: p={p} per step, {n_steps} steps -> {baseline:.4f} ({baseline * 100:.2f}%) end-to-end success")
print()

for c in [0.5, 0.8, 0.95]:
    p_eff = p + q * c * p
    result = p_eff ** n_steps
    print(f"  c={c}: p_eff={p_eff:.5f} -> {result:.4f} ({result * 100:.1f}%)  "
          f"[+{(result - baseline) * 100:.1f} points over the unverified baseline]")

print()
print("The model never got better in any of these rows -- only the fraction")
print("of failures the verifier catches (c) changed. That compounding effect,")
print("multiplied across all 20 steps, is the entire argument for this chapter.")


## Optional -- Run the Same Verified Loop Against a Real Claude Model

The exact same `run_verified_loop`, `V1_BUGGY`, and `TESTS` -- but a live model proposing the fixes instead of a script. This is genuinely live: whatever the real model actually does (how many iterations it takes, whether it ever prematurely declares victory) is what gets measured, via `AnthropicBedrockMantle` (Claude Sonnet, through AWS Bedrock).

In [ ]:
RUN_REAL_LOOP_DEMO = False


def run_real_loop_demo():
    if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
        print("Skipping real loop demo: AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env.")
        return

    real_client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    try:
        success, iters, final_code = run_verified_loop(real_client, MODEL_NAME, V1_BUGGY, TESTS)
    except Exception as exc:
        print(f"Real loop demo failed: {type(exc).__name__}: {exc}")
        return

    print()
    print(f"Real run finished in {iters} iteration(s). success={success}")


if RUN_REAL_LOOP_DEMO:
    run_real_loop_demo()
else:
    print("RUN_REAL_LOOP_DEMO is False -- running in offline/scripted mode only.")
    print("Flip it to True to run this exact verified loop against a real Claude model via Bedrock.")


## Key Takeaways

You built a loop that never trusts a model's own claim of success -- it checks a real verifier every time, including right after an `end_turn` that looks like a legitimate finish. You watched oscillation detection stop a stuck model in 3 iterations instead of burning through a full budget, and best-of-3 pick a genuinely correct fix using the same real verifier rather than a guess. The reliability arithmetic at the end is worth carrying forward: catching more failures (raising $c$) compounds across every step in a chain, the same way caching (Chapter 1) and compaction (Chapter 4) both turned out to hinge on effects that apply repeatedly rather than once.

**Connection forward:** Chapter 7 builds the "planning artifacts" component this chapter leaned on without ever building it -- checklists for progress signals, gates for definitions of done -- into a real, durable plan an agent writes, updates, and is judged against.